# 03 · Bengali continuations, base-model comparison, and literary evaluation

Load one text backbone and toggle its LoRA adapter off/on to compare the original and
adapted model with identical prompts and seeds. This avoids keeping two model copies
in 6 GB VRAM. GPU inference is preferred; CPU FP32 fallback is implemented and needs
several GiB of available system RAM. Generation speed is measured locally, not promised.

Style adaptation does not establish factual literary knowledge or knowledge of every
Tagore work. The example prompts below were written for testing, not attributed to Tagore.
Token limits control generation length, not an exact Bengali word count.


In [ ]:
from pathlib import Path
import sys, json, logging

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "src/tagore_llm").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
from tagore_llm.common import load_config, read_json, read_jsonl, write_json
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
PROJECT = load_config(ROOT)
print("Project:", ROOT)
print("Pinned model:", PROJECT["model_id"], "revision:", PROJECT["revision"])


In [ ]:
LOAD_MODEL = False
RUN_COMPARISON = False
RUN_HELDOUT_EVALUATION = False
SHOW_WIDGET = False
ADAPTER_PATH = None  # None reads models/latest_adapter.json; otherwise use an explicit export directory.
DEVICE = None        # None selects CUDA if available, else CPU. Set "cpu" explicitly if desired.
session = None


## Verify adapter provenance and load

The export carries model/revision identity, run fingerprint and file checksums. Mismatches
are errors. When the training corpus is present, its hash/token audit runs before building
a disk-backed exact-token memorization index. The source metadata remains separate
from the prompts and generated passages.


In [ ]:
from tagore_llm.evaluation import ComparisonSession, GenerationSettings, PROMPTS, display_comparison
if LOAD_MODEL:
    session = ComparisonSession(ROOT, PROJECT, ADAPTER_PATH, DEVICE)
    print("Device:", session.device)
    print("Adapter:", session.adapter_path)
    print(json.dumps(session.loading_report, indent=2))
else:
    print("Model loading disabled; no downloads or weight allocation.")


## Seven kinds of Bengali literary prompts

Test several unrelated prompts and more than one seed. A completion can resemble a
remembered source even when the prompt is novel. Examine both models' outputs,
source-overlap flags and repeated n-gram fraction, alongside your literary judgment.


In [ ]:
for name, text in PROMPTS.items():
    print(name + ":\n" + text + "\n")

PROMPT = PROMPTS["Nature imagery"]
SETTINGS = GenerationSettings(max_new_tokens=160, temperature=0.85, top_p=0.9,
                              top_k=50, repetition_penalty=1.08, seed=42)
SETTINGS.validate()


In [ ]:
if RUN_COMPARISON:
    if session is None:
        raise RuntimeError("Set LOAD_MODEL=True and run the loading cell first.")
    comparison = session.compare(PROMPT, SETTINGS)
    display_comparison(comparison)
    print("Saved ID:", comparison["comparison_id"])
    for key in ["base", "fine_tuned"]:
        print(key, "generated tokens:", comparison[key]["generated_tokens"],
              "repeated 4-gram fraction:", comparison[key]["repeated_4gram_fraction"])
        print(json.dumps(comparison[key]["memorization"], indent=2, ensure_ascii=False))
else:
    print("Generation disabled; no results fabricated.")


## Adjustable generation and saved human ratings

The widget provides prompt, token limit, temperature (0 = greedy), top-p, top-k,
repetition penalty and seed controls. Both models use identical settings. Seeds make
comparisons more controlled, but do not guarantee bit-identical results across hardware
or software versions. Generated text is HTML-escaped before display.

Rate fluency, grammar, coherence, vocabulary, poetic quality, stylistic resemblance and
originality from 1 (poor) to 5 (best). **Unwanted repetition uses 1 = none, 5 = severe.**
Ratings and outputs are saved as JSONL under `evaluations/`; nothing is sent externally.


In [ ]:
if SHOW_WIDGET:
    if session is None:
        raise RuntimeError("Load the model before displaying the evaluation widget.")
    from tagore_llm.evaluation import evaluation_widget
    evaluation_widget(session)
else:
    print("Widget disabled.")


## Held-out NLL and memorization limits

Evaluate both models on the same held-out literary groups. Target-token-weighted NLL
and perplexity measure predictive fit, not poetic merit. The exact-overlap index flags
contiguous matches of at least 24 tokens against stored training sequences and returns
source document IDs for inspection. It may miss shorter, paraphrased or cross-sequence
reproduction. An unflagged result is not proof of originality.


In [ ]:
if RUN_HELDOUT_EVALUATION:
    if session is None:
        raise RuntimeError("Load the model first.")
    losses = session.heldout_losses()
    print(json.dumps(losses, indent=2))
else:
    print("Held-out model evaluation disabled.")


In [ ]:
output_path = ROOT / "evaluations/generations.jsonl"
ratings_path = ROOT / "evaluations/ratings.jsonl"
print("Saved generations:", sum(1 for _ in read_jsonl(output_path)) if output_path.exists() else 0)
print("Saved ratings:", sum(1 for _ in read_jsonl(ratings_path)) if ratings_path.exists() else 0)
# After finishing interactive evaluation, explicitly release GPU memory:
# if session is not None:
#     session.close()
#     session = None
